# テーマA Phase A1：温度側の特徴付け v0.2
> **v0.2**：Driveマウント失敗時に無言でローカルへ切り替わる欠陥を修正
> （Colab上ではマウント必須・失敗したら明示的に停止）。AXIS/FLOORBREAKは
> 救出済みデータをDriveから検出して自動スキップし，残り（p集計・BANDS・
> EXCLUSION・FREQ）のみ実行されます（約15分）。
面除去後の鏡映反対称性（監査②）を5つの角度から特徴付けます。段階制・チェックポイント式
——「すべてのセルを実行」を繰り返してください。

| 段階 | 内容 | 所要 |
|---|---|---|
| AXIS | 10マップ×6構成の最良軸・S(n̂)地形（縮退幅） | 〜15分 |
| FLOORBREAK | ②null拡張：N16×{common,ext}=10⁵・N32×{common,ext}=10⁴ | 〜6時間（分割可） |
| BANDS | 合意軸を凍結→ℓ帯域別の対称成分パワー（データvs null） | 〜10分 |
| EXCLUSION | 領域除外走査（反対称性の担い手マップ） | 〜5分 |
| FREQ | Nofi 4周波数の振幅・軸一致性まとめ | 即時 |

FLOORBREAKが支配的です。夜間は `TIME_BUDGET_MIN=340` 程度に上げてください。

In [ ]:
# ---- 設定（v0.2: マウント失敗は明示停止） ----
TIME_BUDGET_MIN = 110
import os, sys, subprocess, time, json, glob
IN_COLAB = os.path.isdir('/content')
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    if not os.path.isdir('/content/drive/MyDrive'):
        raise RuntimeError('★Driveマウント失敗：セルを再実行し認証を完了してください（ローカル退避はしません）')
    OUTDIR='/content/drive/MyDrive/plane_mirror'
    PHASE2='/content/drive/MyDrive/phase2_null'
    if not os.path.exists(os.path.join(PHASE2,'atlas_raw.csv')):
        raise RuntimeError('★phase2_null/atlas_raw.csv が見つかりません：Drive構成を確認してください')
else:
    OUTDIR='plane_mirror_out'; PHASE2='phase2_null'
os.makedirs(OUTDIR, exist_ok=True)
try: import healpy as hp
except ImportError:
    subprocess.run([sys.executable,'-m','pip','install','-q','healpy']); import healpy as hp
import numpy as np, pandas as pd
if not os.path.isdir('CMBanom'):
    subprocess.run(['git','clone','--depth','1','https://github.com/LauraHerold/CMBanom.git'])
DEADLINE=time.time()+TIME_BUDGET_MIN*60
def left(): return DEADLINE-time.time()
print('OUTDIR =', OUTDIR, '/ atlas OK')

In [ ]:
# ---- 検証済みモジュール ----
open('phase2_core.py','w').write(r'''# -*- coding: utf-8 -*-
"""Phase 2 較正基盤コア（v0.1）
- 処理構成マニフェスト（計画書v1.0 §3準拠）
- CRNマスター実現（synalm, seed 0..999, lmax=128）
- 統計①②③④の null 分布計算（チェックポイント/再開対応）
"""
import os, json, time
import numpy as np, healpy as hp

# ---------- 基本設定 ----------
LMAX_MASTER = 128
NSIM_FULL = 1000
FID_CL_FILE = 'CMBanom/data/real/COM_PowerSpect_CMB-base-plikHM-TTTEEE-lowl-lowE-lensing-minimum-theory_R3.01.txt'
COMMON_MASK_128 = 'CMBanom/data/masks/com_mask_cutoff_0.9_nside_128.fits'

def load_fid_cl(lmax=LMAX_MASTER):
    dat = np.loadtxt(FID_CL_FILE, skiprows=1)
    ll = np.arange(lmax + 1); cl = np.zeros(lmax + 1)
    n = min(lmax - 1, dat.shape[0])
    cl[2:2 + n] = dat[:n, 1] * 2 * np.pi / (ll[2:2 + n] * (ll[2:2 + n] + 1))
    return cl

# ---------- 伝達関数・マスク ----------
PIXWIN_CACHE = 'pixwin_cache'
def pixwin_pad(nside, lmax):
    fn = os.path.join(PIXWIN_CACHE, f'pixel_window_n{nside:04d}.fits')
    if os.path.exists(fn):
        from astropy.io import fits as _f
        with _f.open(fn) as h:
            pw = np.asarray(h[1].data['TEMPERATURE']).ravel()
    else:
        pw = hp.pixwin(nside)   # Colabでは通常経路（初回のみDL）
    return np.pad(pw, (0, max(0, lmax + 1 - len(pw))), mode='edge')[:lmax + 1]

def transfer(nside, smooth, lmax=LMAX_MASTER):
    """構成の伝達関数 b_ℓ p_ℓ。smooth ∈ {'planck','none','fix5deg'}"""
    pw = pixwin_pad(nside, lmax)
    if smooth == 'planck':
        fwhm_arcmin = 640.0 * 16.0 / nside
        return hp.gauss_beam(np.radians(fwhm_arcmin / 60.), lmax=lmax) * pw
    if smooth == 'fix5deg':
        return hp.gauss_beam(np.radians(5.0), lmax=lmax) * pw
    if smooth == 'none':
        return pw.copy()
    raise ValueError(smooth)

def dilate_mask(bad, nside, deg):
    """マスク（bad=True）を約deg度拡張（近傍膨張の反復）"""
    pixsize_deg = np.degrees(hp.nside2resol(nside))
    n_iter = max(1, int(np.ceil(deg / pixsize_deg)))
    bad = bad.copy()
    for _ in range(n_iter):
        idx = np.where(bad)[0]
        nb = hp.get_all_neighbours(nside, idx)
        bad[nb[nb >= 0]] = True
    return bad

def _dilate_n(bad, nside, n_iter):
    bad = bad.copy()
    for _ in range(n_iter):
        idx = np.where(bad)[0]
        nb = hp.get_all_neighbours(nside, idx)
        bad[nb[nb >= 0]] = True
    return bad

def _erode_n(bad, nside, n_iter):
    return ~_dilate_n(~bad, nside, n_iter)

def make_mask(nside, level):
    """level ∈ {'full','common','ext'} → bool（True=使用画素）
    ext = Planck 2015 XVI Table 12方式：共通マスクの拡散（銀河）成分のみを5°拡張し
          点源穴は拡張しない（補助マスク規定）。128で構成してから縮退。"""
    npix = hp.nside2npix(nside)
    if level == 'full':
        return np.ones(npix, bool)
    m128 = hp.read_map(COMMON_MASK_128)
    if level == 'common':
        return hp.ud_grade(m128, nside) >= 0.9
    if level == 'ext':
        bad128 = m128 < 0.5
        pix_deg = np.degrees(hp.nside2resol(128))          # ≈0.46°
        n_open = 2                                          # 開演算で点源(≲1°)を除去
        diffuse = _dilate_n(_erode_n(bad128, 128, n_open), 128, n_open)
        n5 = int(np.ceil(5.0 / pix_deg))                    # 5°膨張
        bad_ext = bad128 | _dilate_n(diffuse, 128, n5)
        return hp.ud_grade((~bad_ext).astype(float), nside) >= 0.9
    raise ValueError(level)

# ---------- CRNマスター実現 ----------
def gen_master_alm(outfile, nsim=NSIM_FULL, lmax=LMAX_MASTER):
    if os.path.exists(outfile):
        return np.load(outfile)['alms']
    cl = load_fid_cl(lmax)
    alms = np.empty((nsim, hp.Alm.getsize(lmax)), dtype=np.complex128)
    for s in range(nsim):
        np.random.seed(s)
        alms[s] = hp.synalm(cl, lmax=lmax)
    np.savez_compressed(outfile, alms=alms, lmax=lmax, nsim=nsim,
                        note='CRN master: fiducial PR3 bestfit, seeds 0..nsim-1')
    return alms

def config_maps(alms, nside, smooth, route='harmonic', lmax=LMAX_MASTER):
    """マスター実現→構成マップ群 (nsim, npix)"""
    bl = transfer(nside, smooth, lmax)
    npix = hp.nside2npix(nside)
    out = np.empty((alms.shape[0], npix))
    if route == 'harmonic':
        for s in range(alms.shape[0]):
            out[s] = hp.alm2map(hp.almxfl(alms[s], bl), nside)
    elif route == 'udgrade':   # 高解像度で実体化→画素平均（quadrature軸）
        nhi = min(4 * nside, 64)
        bl_hi = transfer(nside, smooth, lmax) / pixwin_pad(nside, lmax) * pixwin_pad(nhi, lmax)
        for s in range(alms.shape[0]):
            out[s] = hp.ud_grade(hp.alm2map(hp.almxfl(alms[s], bl_hi), nhi), nside)
    else:
        raise ValueError(route)
    return out

# ---------- 統計②：鏡映パリティ（マスク対応） ----------
class MirrorStat:
    def __init__(self, nside, mask):
        npix = hp.nside2npix(nside)
        vecs = np.array(hp.pix2vec(nside, np.arange(npix))).T
        R = np.empty((npix, npix), dtype=np.int32)
        for i in range(npix):
            n = vecs[i]
            refl = vecs - 2.0 * np.outer(vecs @ n, n)
            R[i] = hp.vec2pix(nside, refl[:, 0], refl[:, 1], refl[:, 2])
        self.R = R
        self.valid = mask[None, :] & mask[R]          # 双方が有効な対のみ
        self.cnt = np.maximum(self.valid.sum(axis=1), 1)
        self.mask = mask

    def min_S(self, mp, chunk=1024, mondip=True):
        if mondip:
            mp = np.asarray(hp.remove_dipole(hp.ma(np.where(self.mask, mp, hp.UNSEEN))))
        mp = np.where(self.mask, mp, 0.0).astype(np.float32)
        nd = self.R.shape[0]
        Sp = np.empty(nd, np.float32); Sm = np.empty(nd, np.float32)
        for a in range(0, nd, chunk):
            Tr = mp[self.R[a:a + chunk]]
            V = self.valid[a:a + chunk]
            Sp[a:a + chunk] = np.sum(V * (0.5 * (mp[None, :] + Tr)) ** 2, axis=1) / self.cnt[a:a + chunk]
            Sm[a:a + chunk] = np.sum(V * (0.5 * (mp[None, :] - Tr)) ** 2, axis=1) / self.cnt[a:a + chunk]
        return float(Sp.min()), float(Sm.min())

# ---------- 統計①：R/D（QML / MASTER / naive fsky） ----------
def C_pm(cl_from2, lmax):
    ell = np.arange(2, lmax + 1)
    Dl = ell * (ell + 1.) / (2 * np.pi) * cl_from2[:lmax - 1]
    ev = (ell % 2 == 0)
    return Dl[ev].sum() / (lmax - 1), Dl[~ev].sum() / (lmax - 1)

def RD_traj(cl_from2, lmaxes):
    out = np.empty((len(lmaxes), 2))
    for i, L in enumerate(lmaxes):
        p, m = C_pm(cl_from2, L)
        out[i] = (p / m, p - m)
    return out

def lmax_est_of(nside):
    return int(min(40, 2.5 * nside))

# ---------- 統計③：多重極ベクトル（polyMV非依存・性質テスト済） ----------
from scipy.special import gammaln as _gln

def multipole_vectors(alm, lmax, ell):
    a = np.zeros(2*ell+1, dtype=complex)
    for m in range(0, ell+1):
        v = alm[hp.Alm.getidx(lmax, ell, m)]
        a[ell+m] = v
        if m: a[ell-m] = (-1)**m * np.conj(v)
    k = np.arange(2*ell+1)
    logC = _gln(2*ell+1) - _gln(k+1) - _gln(2*ell-k+1)
    c = np.sqrt(np.exp(logC)) * a
    roots = np.roots(c[::-1])
    th = 2*np.arctan(np.abs(roots)); ph = np.angle(roots) + np.pi
    v = np.stack([np.sin(th)*np.cos(ph), np.sin(th)*np.sin(ph), np.cos(th)], 1)
    v = np.where(v[:, 2:3] >= 0, v, -v)
    keep = []
    for i in range(len(v)):
        if not any(np.dot(v[i], v[j]) > 0.999 for j in keep): keep.append(i)
    return v[keep[:ell]]

def stat3_SQO(maps, mask, lmax_alm=8):
    """素朴カットスカイalm経路（第1弾で検証済みの規約）でS_QO"""
    vals = np.empty(maps.shape[0])
    fmask = mask.astype(float)
    for s in range(maps.shape[0]):
        alm = hp.map2alm(maps[s]*fmask, lmax=lmax_alm, iter=3)
        v2 = multipole_vectors(alm, lmax_alm, 2)
        v3 = multipole_vectors(alm, lmax_alm, 3)
        w2 = np.cross(v2[0], v2[1])
        w3 = [np.cross(v3[i], v3[j]) for i in range(3) for j in range(i+1, 3)]
        vals[s] = np.mean([abs(np.dot(w2, w)) for w in w3])
    return vals

# ---------- 実行系（チェックポイント） ----------
def null_dir(base, cfg_id):
    d = os.path.join(base, cfg_id); os.makedirs(d, exist_ok=True); return d

def true_varlvmap(lvmaps, lvmask, mean_lvmap):
    """正しい逆分散重み用の規格化分散（リポジトリ版get_varlvmapは定数(N-1)²/Nになるバグ）"""
    return np.where(lvmask == 1.,
                    np.mean((lvmaps - mean_lvmap) ** 2, axis=0) / np.maximum(mean_lvmap, 1e-30) ** 2,
                    1.)

def run_stat2(base, cfg_id, maps, nside, mask, chunk_ckpt=100, mondip=True):
    """②のnull分布（部分保存・再開対応）"""
    d = null_dir(base, cfg_id); fn = os.path.join(d, 'stat2.npz')
    done = 0; minSp = []; minSm = []
    if os.path.exists(fn):
        z = np.load(fn)
        if z['complete']: return
        minSp, minSm, done = list(z['minSp']), list(z['minSm']), int(z['done'])
    ms = MirrorStat(nside, mask)
    for s in range(done, maps.shape[0]):
        sp, sm = ms.min_S(maps[s], mondip=mondip)
        minSp.append(sp); minSm.append(sm)
        if (s + 1) % chunk_ckpt == 0 or s == maps.shape[0] - 1:
            np.savez(fn, minSp=minSp, minSm=minSm, done=s + 1,
                     complete=(s == maps.shape[0] - 1))
    return np.array(minSp), np.array(minSm)
''')
open('plane_mirror.py','w').write(r'''# -*- coding: utf-8 -*-
"""plane_mirror.py — テーマA: 鏡映反対称性の高速バッチ評価
MirrorStat（phase2_core）と厳密同一の定義で，方向走査をマップ束一括化。
検証済み（2026-08-15）: MirrorStatとの max相対差 1.2e-6（float32水準）。
計時: N16 33ms/マップ（10^5=55分）, N32 0.9s/マップ（10^4=2.5時間）。
"""
import numpy as np
import healpy as hp


class MirrorBatch:
    def __init__(self, ms):
        self.R, self.valid, self.cnt, self.mask = ms.R, ms.valid, ms.cnt, ms.mask

    def min_S_batch(self, maps, mondip=True, return_argmin=False):
        B = maps.shape[0]
        T = np.empty((maps.shape[1], B), np.float32)
        for b in range(B):
            m = maps[b]
            if mondip:
                m = np.asarray(hp.remove_dipole(hp.ma(np.where(self.mask, m, hp.UNSEEN))))
            T[:, b] = np.where(self.mask, m, 0.0)
        nd = self.R.shape[0]
        minSp = np.full(B, np.inf, np.float32); minSm = np.full(B, np.inf, np.float32)
        argp = np.zeros(B, np.int32); argm = np.zeros(B, np.int32)
        for d in range(nd):
            v = self.valid[d]
            if not v.any():
                continue
            Tr = T[self.R[d]]
            Tv, Trv = T[v], Tr[v]
            Sp = np.einsum('ib,ib->b', 0.5 * (Tv + Trv), 0.5 * (Tv + Trv)) / self.cnt[d]
            Sm = np.einsum('ib,ib->b', 0.5 * (Tv - Trv), 0.5 * (Tv - Trv)) / self.cnt[d]
            better = Sp < minSp
            if return_argmin:
                argp = np.where(better, d, argp)
            minSp = np.where(better, Sp, minSp)
            better = Sm < minSm
            if return_argmin:
                argm = np.where(better, d, argm)
            minSm = np.where(better, Sm, minSm)
        if return_argmin:
            return minSp, minSm, argp, argm
        return minSp, minSm


def axis_lb(nside, d):
    """方向画素番号 → 銀経緯 (l, b) [deg]"""
    th, ph = hp.pix2ang(nside, int(d))
    return float(np.degrees(ph)), float(90.0 - np.degrees(th))


class FixedAxisMirror:
    """凍結軸 d* での対称成分解析（O(npix)/マップ）"""
    def __init__(self, ms, d_star):
        self.r = ms.R[d_star]
        self.v = ms.valid[d_star]
        self.cnt = ms.cnt[d_star]
        self.mask = ms.mask

    def S_plus(self, mp, mondip=True):
        if mondip:
            mp = np.asarray(hp.remove_dipole(hp.ma(np.where(self.mask, mp, hp.UNSEEN))))
        T = np.where(self.mask, mp, 0.0)
        s = 0.5 * (T + T[self.r])
        return float(np.sum(self.v * s * s) / self.cnt)

    def S_plus_batch(self, maps, mondip=True):
        return np.array([self.S_plus(maps[b], mondip) for b in range(maps.shape[0])])

    def band_decompose(self, alm128, nside, transfer_fl, bands, mondip=True):
        """ソースalm(lmax128)を帯域分解し，帯域別S⁺と全帯域和・交差項を返す"""
        import numpy as _np
        LMAX = hp.Alm.getlmax(len(alm128))
        ell = _np.arange(LMAX + 1)
        out = {}
        s_parts = []
        for (l0, l1) in bands:
            w = ((ell >= l0) & (ell <= l1)).astype(float) * transfer_fl
            mb = hp.alm2map(hp.almxfl(alm128.copy(), w), nside)
            if mondip and l0 <= 1:
                pass
            T = _np.where(self.mask, mb, 0.0)
            if mondip:
                T = _np.where(self.mask,
                              _np.asarray(hp.remove_dipole(hp.ma(_np.where(self.mask, mb, hp.UNSEEN)))), 0.0)
            s = 0.5 * (T + T[self.r])
            s_parts.append(s)
            out[f'S{l0}_{l1}'] = float(_np.sum(self.v * s * s) / self.cnt)
        stot = _np.sum(s_parts, axis=0)
        out['S_sum_bands'] = float(_np.sum(self.v * stot * stot) / self.cnt)
        return out

    def exclusion_scan(self, mp, nside_scan=8, radius_deg=15.0, mondip=True):
        """半径radius_degの円盤を各走査位置で（鏡映相手も対称に）除外した際の
        ln S⁺ の変化 Δ(q) を返す（正＝除外で対称性が増える＝その領域が反対称の担い手）"""
        if mondip:
            mp = np.asarray(hp.remove_dipole(hp.ma(np.where(self.mask, mp, hp.UNSEEN))))
        T = np.where(self.mask, mp, 0.0)
        s = 0.5 * (T + T[self.r])
        base_num = np.sum(self.v * s * s)
        base = base_num / self.cnt
        npix_scan = hp.nside2npix(nside_scan)
        nside_map = hp.npix2nside(len(T))
        delta = np.zeros(npix_scan)
        for q in range(npix_scan):
            vec = hp.pix2vec(nside_scan, q)
            disc = hp.query_disc(nside_map, vec, np.radians(radius_deg))
            ex = np.zeros(len(T), bool); ex[disc] = True
            ex = ex | ex[self.r]                       # 対称除外
            v2 = self.v & ~ex
            c2 = max(v2.sum(), 1)
            S2 = np.sum(v2 * s * s) / c2
            delta[q] = np.log(S2 / base) if S2 > 0 else 0.0
        return delta, base


def with_mask(ms, mask):
    """R表を共有して別マスクのMirrorStat相当を作る（N32のR再構築20秒を節約）"""
    obj = type(ms).__new__(type(ms))
    obj.R = ms.R
    obj.mask = mask
    obj.valid = mask[None, :] & mask[ms.R]
    obj.cnt = np.maximum(obj.valid.sum(axis=1), 1)
    return obj


def scan_S(ms, mp, mondip=True):
    """1マップの全方向S±(n̂)地形を返す（軸の縮退・地形幅の解析用）"""
    if mondip:
        mp = np.asarray(hp.remove_dipole(hp.ma(np.where(ms.mask, mp, hp.UNSEEN))))
    T = np.where(ms.mask, mp, 0.0).astype(np.float32)
    nd = ms.R.shape[0]
    Sp = np.empty(nd, np.float32); Sm = np.empty(nd, np.float32)
    for d in range(nd):
        v = ms.valid[d]
        Tr = T[ms.R[d]]
        Sp[d] = np.sum(v * (0.5 * (T + Tr)) ** 2) / ms.cnt[d]
        Sm[d] = np.sum(v * (0.5 * (T - Tr)) ** 2) / ms.cnt[d]
    return Sp, Sm


def axis_sep_deg(nside, d1, d2):
    """2軸の分離角[deg]（鏡映面法線は±同一視 → 90°超は補角）"""
    v1 = np.array(hp.pix2vec(nside, int(d1)))
    v2 = np.array(hp.pix2vec(nside, int(d2)))
    ang = np.degrees(np.arccos(np.clip(abs(v1 @ v2), -1, 1)))
    return float(ang)
''')
import importlib, phase2_core as p2, plane_mirror as pm
for m in (p2,pm): importlib.reload(m)
print('モジュール準備OK')

In [ ]:
# ---- ソースと基本オブジェクト ----
LMAX=128; CL=p2.load_fid_cl()
from phase2_core import pixwin_pad
T_SRC = hp.gauss_beam(np.radians(1.0), lmax=LMAX)*pixwin_pad(128, LMAX)
SOURCES={}
repo_maps={'PR3_Commander':'commander','PR3_NILC':'nilc','PR3_SEVEM':'sevem','PR3_SMICA':'smica',
           'Nofi_70GHz':'cleaned_70GHz_v9','Nofi_94GHz':'cleaned_94GHz_v9',
           'Nofi_100GHz':'cleaned_100GHz_v9','Nofi_143GHz':'cleaned_143GHz_v9'}
for name,f in repo_maps.items():
    SOURCES[name]=hp.read_map(f'CMBanom/data/real/map_{f}_nside_128.fits')
for meth in ['sevem','commander']:
    c=os.path.join(PHASE2,'sources',f'npipe_{meth}_128.fits')
    if os.path.exists(c): SOURCES[f'PR4_{meth.capitalize()}']=hp.read_map(c)
def data_map(name,nside):
    alm=hp.map2alm(SOURCES[name],lmax=LMAX)
    fl=p2.transfer(nside,'planck',LMAX)/np.maximum(T_SRC,1e-12)
    return hp.alm2map(hp.almxfl(alm,fl),nside)
MS={}   # (nside)->基準MirrorStat(full), マスク差し替えはwith_mask
def get_ms(nside, maskl):
    if nside not in MS:
        MS[nside]=p2.MirrorStat(nside, p2.make_mask(nside,'full'))
    if maskl=='full': return MS[nside]
    return pm.with_mask(MS[nside], p2.make_mask(nside, maskl))
print('ソース:', list(SOURCES))

In [ ]:
# ---- 段階1 AXIS：最良軸と地形 ----
axfn=os.path.join(OUTDIR,'a1_axes.csv')
if os.path.exists(axfn):
    axes=pd.read_csv(axfn); print('AXIS: 済')
else:
    rows=[]
    for nside in [16,32]:
        for K in ['full','common','ext']:
            ms=get_ms(nside,K)
            for name in SOURCES:
                m=data_map(name,nside)
                Sp,Sm=pm.scan_S(ms,m)
                d=int(Sp.argmin()); l,b=pm.axis_lb(nside,d)
                width=float((Sp<1.05*Sp.min()).mean())   # 地形の縮退幅（5%以内の方向割合）
                rows.append(dict(map=name,nside=nside,mask=K,axis_pix=d,l=l,b=b,
                                 S_min=float(Sp.min()),S_med=float(np.median(Sp)),width5=width))
            print(f'AXIS N{nside}/{K} 完了')
    axes=pd.DataFrame(rows); axes.to_csv(axfn,index=False)
# 軸の一致性サマリ
ref=axes[(axes['map']=='PR3_SMICA')&(axes.nside==16)&(axes['mask']=='common')]
if len(ref):
    d0=int(ref.axis_pix.iloc[0])
    print('PR3_SMICA/N16/common の軸を基準にした分離角[deg]:')
    for _,r in axes[(axes.nside==16)&(axes['mask']=='common')].iterrows():
        sep=pm.axis_sep_deg(16,d0,int(r.axis_pix))
        print(f"  {r['map']:15s} (l,b)=({r.l:5.0f},{r.b:+4.0f})  sep={sep:5.1f}°  width5={r.width5:.3f}")

In [ ]:
# ---- 段階2 FLOORBREAK：②null拡張（チェックポイント） ----
FB=os.path.join(OUTDIR,'floorbreak'); os.makedirs(FB,exist_ok=True)
TARGETS=[('N16_Splanck_Kcommon_mdON_harm',16,'common',100_000,5000),
         ('N16_Splanck_Kext_mdON_harm',16,'ext',100_000,5000),
         ('N32_Splanck_Kcommon_mdON_harm',32,'common',10_000,500),
         ('N32_Splanck_Kext_mdON_harm',32,'ext',10_000,500)]
for cid,nside,K,NBIG,CHUNK in TARGETS:
    fn=os.path.join(FB,cid+'.npz'); done=0; Sp=Sm=None
    if os.path.exists(fn):
        z=np.load(fn); Sp,Sm,done=z['Sp'],z['Sm'],int(z['done'])
        if done>=NBIG: print(f'[{cid}] 完了({done})'); continue
    if left()<0: print('時間予算到達'); break
    ms=get_ms(nside,K); mb=pm.MirrorBatch(ms)
    bl=p2.transfer(nside,'planck')
    while done<NBIG and left()>0:
        n=min(CHUNK,NBIG-done); t0=time.time()
        maps=np.empty((n,hp.nside2npix(nside)),np.float32)
        for i,s in enumerate(range(done,done+n)):
            np.random.seed(s)
            maps[i]=hp.alm2map(hp.almxfl(hp.synalm(CL,lmax=LMAX),bl),nside)
        sp,sm=mb.min_S_batch(maps)
        Sp = sp if Sp is None else np.concatenate([Sp,sp])
        Sm = sm if Sm is None else np.concatenate([Sm,sm])
        done+=n; np.savez(fn,Sp=Sp,Sm=Sm,done=done)
        print(f'  [{cid}] {done}/{NBIG} (+{time.time()-t0:.0f}s, 残り{left()/60:.0f}分)')
    # 同一アンサンブル検証（監査の保存済みstat2 nullが見つかれば）
    for cand in glob.glob(os.path.join(PHASE2,cid,'stat2*.npz'))+glob.glob(os.path.join(PHASE2,cid+'_stat2.npz')):
        try:
            zz=np.load(cand)
            key=[k for k in zz.files if 'p' in k.lower() or 'S' in k][0]
            ref=np.asarray(zz[key]).ravel()[:1000]
            dev=np.abs(Sp[:len(ref)]/ref-1).max()
            print(f'  同一性検証({os.path.basename(cand)}): max相対差={dev:.1e}')
        except Exception as e: print('  同一性検証スキップ:',e)
# データp再計算
atl=pd.read_csv(os.path.join(PHASE2,'atlas_raw.csv'))
for c in ['map','config','stat','sub']: atl[c]=atl[c].astype(str)
from scipy.stats import beta as _beta
rows=[]
for cid,nside,K,NBIG,_ in TARGETS:
    fn=os.path.join(FB,cid+'.npz')
    if not os.path.exists(fn): continue
    z=np.load(fn); S=z['Sp']; N=int(z['done'])
    sel=atl[(atl.config==cid)&(atl.stat=='stat2')&(atl['sub']=='minS+')]
    for _,r in sel.iterrows():
        k=int((S<r['value']).sum()); p=max(k,1)/N
        lo=_beta.ppf(0.16,k+0.5,N-k+0.5); hi=_beta.ppf(0.84,k+1,N-k)
        rows.append(dict(config=cid,map=r['map'],N=N,k=k,p=p,ci68=f'[{lo:.2e},{hi:.2e}]'))
        print(f"{cid.replace('_mdON_harm','')[:22]:22s} {r['map']:15s} p={p:.2e} (k={k}/{N})")
pd.DataFrame(rows).to_csv(os.path.join(OUTDIR,'a1_floorbreak.csv'),index=False)

In [ ]:
# ---- 段階3 BANDS：合意軸凍結→帯域分解 ----
bfn=os.path.join(OUTDIR,'a1_bands.csv')
axes=pd.read_csv(os.path.join(OUTDIR,'a1_axes.csv'))
# 合意軸：PR3×N16/commonの軸の単位ベクトル平均（±同一視）→最近接画素
sel=axes[(axes.nside==16)&(axes['mask']=='common')&axes['map'].str.startswith('PR3')]
vecs=[]
v0=np.array(hp.pix2vec(16,int(sel.axis_pix.iloc[0])))
for _,r in sel.iterrows():
    v=np.array(hp.pix2vec(16,int(r.axis_pix)))
    vecs.append(v if v@v0>=0 else -v)
vc=np.mean(vecs,axis=0); vc/=np.linalg.norm(vc)
D_STAR=int(hp.vec2pix(16,*vc)); L0,B0=pm.axis_lb(16,D_STAR)
print(f'合意軸（凍結・A2で使用）: pix={D_STAR}, (l,b)=({L0:.1f}°,{B0:.1f}°)')
json.dump(dict(axis_pix=D_STAR,l=L0,b=B0,rule='PR3 x N16/common unit-vector mean'),
          open(os.path.join(OUTDIR,'a1_consensus_axis.json'),'w'))
if os.path.exists(bfn):
    print('BANDS: 済')
else:
    ms=get_ms(16,'common'); fa=pm.FixedAxisMirror(ms,D_STAR)
    BANDS=[(2,4),(5,8),(9,16),(17,32),(33,64)]
    fl=p2.transfer(16,'planck',LMAX)
    ALMS=p2.gen_master_alm(os.path.join(PHASE2,'crn_master_1000.npz'), nsim=1000)
    null={f'S{a}_{b}':[] for a,b in BANDS}
    t0=time.time()
    for s in range(1000):
        r=fa.band_decompose(ALMS[s].copy(),16,fl,BANDS)
        for k in null: null[k].append(r[k])
    print(f'null帯域分解 1000実現 {time.time()-t0:.0f}s')
    rows=[]
    for name in SOURCES:
        alm=hp.almxfl(hp.map2alm(SOURCES[name],lmax=LMAX),1.0/np.maximum(T_SRC,1e-12))
        r=fa.band_decompose(alm,16,fl,BANDS)
        for a,b in BANDS:
            k=f'S{a}_{b}'; nl=np.array(null[k])
            p=max((nl<r[k]).sum(),1)/1000
            rows.append(dict(map=name,band=k,S_data=r[k],S_null_med=float(np.median(nl)),
                             ratio=r[k]/np.median(nl),p_lower=p))
    pd.DataFrame(rows).to_csv(bfn,index=False)
    x=pd.DataFrame(rows)
    print(x[x['map']=='PR3_SMICA'][['band','ratio','p_lower']].to_string(index=False))

In [ ]:
# ---- 段階4 EXCLUSION＋段階5 FREQ ----
exfn=os.path.join(OUTDIR,'a1_exclusion.npz')
axesj=json.load(open(os.path.join(OUTDIR,'a1_consensus_axis.json')))
ms=get_ms(16,'common'); fa=pm.FixedAxisMirror(ms,int(axesj['axis_pix']))
if not os.path.exists(exfn):
    deltas={}; 
    for name in SOURCES:
        d,base=fa.exclusion_scan(data_map(name,16),nside_scan=8,radius_deg=15.0)
        deltas[name]=d
    # null参照スケール（CRN 50実現の|Δ|中央値）
    ALMS=p2.gen_master_alm(os.path.join(PHASE2,'crn_master_1000.npz'), nsim=50)
    nd=[]
    for s in range(50):
        m=hp.alm2map(hp.almxfl(ALMS[s].copy(),p2.transfer(16,'planck')),16)
        d,_=fa.exclusion_scan(m); nd.append(np.abs(d))
    np.savez(exfn, null_absmed=np.median(nd,axis=0), **deltas)
    print('EXCLUSION 保存。データ|Δ|max:',
          {n[:8]: round(float(np.abs(deltas[n]).max()),3) for n in list(deltas)[:4]})
else: print('EXCLUSION: 済')
# FREQ: Nofi一致性
axes=pd.read_csv(os.path.join(OUTDIR,'a1_axes.csv'))
fb=pd.read_csv(os.path.join(OUTDIR,'a1_floorbreak.csv')) if os.path.exists(os.path.join(OUTDIR,'a1_floorbreak.csv')) else None
print('\n== FREQ: Nofi 4周波数の一致性（N16/common）==')
nofi=axes[(axes.nside==16)&(axes['mask']=='common')&axes['map'].str.startswith('Nofi')]
d0=int(nofi.axis_pix.iloc[0])
for _,r in nofi.iterrows():
    print(f"  {r['map']:12s} S_min={r.S_min:7.1f} (l,b)=({r.l:5.0f},{r.b:+4.0f}) "
          f"sep(70GHz基準)={pm.axis_sep_deg(16,d0,int(r.axis_pix)):.1f}°")
print('\nA1 全段階の出力: a1_axes.csv / a1_floorbreak.csv / a1_bands.csv / '
      'a1_exclusion.npz / a1_consensus_axis.json → 添付してください')

## 完了後
5つの出力ファイルを添付してください。A1解析（軸一致性の判定・帯域の担い手・
除外走査マップの図化）を行い，フォーキャスト→GO/NO-GO判定→A2事前登録へ進みます。